In [32]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("MySession").getOrCreate()
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

print(spark.version)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

3.0.1-amzn-0

In [45]:
import json
import pandas as pd
from pyspark.sql.functions import col, udf, to_timestamp, desc, year, count, round, monotonically_increasing_id, concat, lit
from pyspark.sql.functions import from_unixtime, from_json, explode
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import pyspark.sql.functions as F
import re

from pyspark.sql.functions import lower, regexp_replace, from_unixtime, year, to_date

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [46]:
df = spark.read.json("s3://1313131buckey/politics_submissions.zst")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [47]:
df_clean = df.select(
    "title",
    "created_utc",
    "author",
    "id",
    "link_flair_text",
    "url",
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [48]:
df_clean.show(2)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------+-----------+------------+-----+---------------+--------------------+
|               title|created_utc|      author|   id|link_flair_text|                 url|
+--------------------+-----------+------------+-----+---------------+--------------------+
|Republican hopefu...| 1186378251|         db2|2cnih|           null|http://in.reuters...|
|Poem From Guantanamo| 1186378432|goldfish1man|2cnj0|           null|http://www.dailyk...|
+--------------------+-----------+------------+-----+---------------+--------------------+
only showing top 2 rows

In [49]:
df_clean = df_clean.withColumn("year", year(from_unixtime(col("created_utc"))))
# df_2021 = df_clean.filter(col("year") == 2021)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [50]:
# Define keywords
keywords = ['abortion', 'abortions', 'roe v wade', 'roe v. wade', 'dobbs v. jackson', 'dobbs v jackson', 
            'roe vs wade', 'dobbs vs jackson', 'roe vs. wade', 'dobbs vs. jackson', 'roe',
            'pro life', 'pro choice', 'prolife', 'prochoice', 'pro-life', 'pro-choice']

# Create regex pattern for case-insensitive search, handling punctuation
pattern = r"\b(" + "|".join(r"\b" + re.escape(keyword) + r"\b" for keyword in keywords) + r")\b"

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [51]:
print(pattern)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

\b(\babortion\b|\babortions\b|\broe\ v\ wade\b|\broe\ v\.\ wade\b|\bdobbs\ v\.\ jackson\b|\bdobbs\ v\ jackson\b|\broe\ vs\ wade\b|\bdobbs\ vs\ jackson\b|\broe\ vs\.\ wade\b|\bdobbs\ vs\.\ jackson\b|\broe\b|\bpro\ life\b|\bpro\ choice\b|\bprolife\b|\bprochoice\b|\bpro\-life\b|\bpro\-choice\b)\b

In [52]:
df_filtered = df_clean.filter(
    lower(col("title")).rlike(pattern)
)

df_filtered = df_filtered.withColumn("time", from_unixtime(df_filtered['created_utc'].cast("bigint")))
df_filtered = df_filtered.drop("created_utc", "year")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
df_filtered.write.parquet("s3://1313131buckey/politics_submissions.parquet", mode="overwrite")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
test = spark.read.parquet("s3://1313131buckey/politics_submissions.parquet")

In [ ]:
test.show(5, truncate=False)

In [ ]:
test.count()
